# 03 - Search Methods: BM25

This notebook builds the first search method that we will try, which is keyword search with BM25, and measures how often it finds the evidence page on the 30 dev questions. The 120 test questions are not used here.

It produces two things:

- `data/processed/pages.jsonl`, the text of every page of the 84 PDFs, which later notebooks reuse
- `results/dev_retrieval_bm25.json`, the pages BM25 found for each dev question, which will be read in model comparison notebooks

The search code can be found in  `src/financebench_rag/` as a package, which needs to be installed with `pip install -e .` from the repo root.

In [1]:
import json
from collections import Counter

import bm25s
import pandas as pd

from financebench_rag.data import REPO, extract_pages, gold_pages, load_doc_info, load_pages, load_questions, load_split
from financebench_rag.evaluate import evaluate_retrieval
from financebench_rag.retrieval import BM25Search, build_chunks

questions = load_questions()
dev_ids = set(load_split()["dev"])
dev = [q for q in questions if q["financebench_id"] in dev_ids]
doc_info = load_doc_info()
print(f"{len(dev)} dev questions")

30 dev questions


## 1. Extract the page text

The text of every page is extracted with PyMuPDF once and saved, so that no later step has to open the PDFs again. Each line of `pages.jsonl` holds one page with its document name, its 0-based page number and its text.

In [2]:
extract_pages()
pages = load_pages()
print(f"{len(pages)} pages, {sum(not p['text'].strip() for p in pages)} of them without any text")

12013 pages, 65 of them without any text


## 2. Cut the pages into chunks

We cut each page into chunks of 250 words. Neighbouring chunks share 50 words (overlap), so a sentence that sits on a cut still appears whole in one of them.

The search scores chunks instead of whole pages. 

For every chunk, we add a prefix that holds:
- the company name
- the document type
- the year of the report, written twice as `FY2018` and as `2018` since BM25 only matches whole words

because a chunk from the middle of a page usually mentions none of these but the questions ask for them.

In [3]:
chunks = build_chunks(pages, doc_info, size=250, overlap=50, prefix=True)
print(f"{len(chunks)} chunks from {len(pages)} pages\n")
print(chunks[5000]["text"][:400])

33998 chunks from 12013 pages

Amcor 10k FY2020 2020
Guarantor in respect of, such Specified Indebtedness, pursuant to which such Subsidiary will guarantee the Notes issued under the indenture on the same terms and subject to the same conditions and limitations as set forth in the indenture. Any supplemental indenture entered into in accordance with the indenture in connection with the provision of a Guarantee by an additional 


## 3. Search with BM25

BM25 scores a chunk by the words it shares with the question. Three things decide the score:

- - a shared word counts more when it is rare in the corpus, so "analyst" (in 24 chunks) counts far more than "year" (in 7,832 chunks), as the table in section 6 shows
- a word that appears several times in a chunk counts more, but with diminishing returns
- long chunks get a small penalty, because they contain more words by chance

The search returns pages, because the evidence labels are pages. Each page gets the score of its best chunk. The cell below shows the top 5 pages for one dev question next to its evidence page.

In [4]:
search = BM25Search(chunks)

example = next(q for q in dev if q["financebench_id"] == "financebench_id_01482")
print("Q:", example["question"])
print("evidence page:", sorted(gold_pages(example)))
print("top 5 pages:")
for page in search.search(example["question"], k=5):
    print("  ", page)

Q: At the Pepsico AGM held on May 3, 2023, what was the outcome of the shareholder vote on the shareholder proposal for a congruency report by Pepsico on net-zero emissions policies?
evidence page: [('PEPSICO_2023_8K_dated-2023-05-05', 3)]
top 5 pages:
   ('PEPSICO_2023_8K_dated-2023-05-05', 3)
   ('FOOTLOCKER_2022_8K_dated-2022-05-20', 1)
   ('AMCOR_2023_10K', 123)
   ('3M_2022_10K', 125)
   ('AMCOR_2023_10K', 120)


## 4. Scores on the dev questions

A result counts as a hit when any evidence page of the question appears in it, since some questions have evidence on more than one page. The metrics are:

- **recall@1** is the share of questions whose first result is an evidence page.
- **recall@5** is the share of questions with an evidence page somewhere in the top 5.
- **mrr@10** is the mean reciprocal rank. A question scores 1 if the evidence page is ranked first, 1/2 if second, 1/3 if third, and 0 if it is not in the top 10.
- **doc_recall@5** is the share of questions where at least one of the top 5 pages comes from the right report, even if it is the wrong page.

In [5]:
results = {q["financebench_id"]: search.search(q["question"], k=10) for q in dev}
evaluate_retrieval(results, dev)

{'recall@1': 0.1, 'recall@5': 0.133, 'mrr@10': 0.111, 'doc_recall@5': 0.6}

In [6]:
# The same recall@5, split by question type.
hits = pd.DataFrame({
    "question_type": [q["question_type"] for q in dev],
    "hit@5": [any(page in gold_pages(q) for page in results[q["financebench_id"]][:5]) for q in dev],
})
hits.groupby("question_type")["hit@5"].agg(["sum", "count"])

,sum,count
question_type,,
domain-relevant,1,10
metrics-generated,0,10
novel-generated,3,10


The table below lists the dev questions whose evidence page is in BM25's top 10, with the rank of the evidence page.

In [13]:
pd.set_option("display.max_colwidth", None)

found = []
for q in dev:
    ranked = results[q["financebench_id"]]
    rank = next((i + 1 for i, page in enumerate(ranked) if page in gold_pages(q)), None)
    if rank:
        found.append({"rank": rank, "question_type": q["question_type"], "question": q["question"]})
pd.DataFrame(found)

,rank,question_type,question
0,5,novel-generated,What was the key agenda of the AMCOR's 8k filing dated 1st July 2022?
1,1,domain-relevant,Which debt securities are registered to trade on a national securities exchange under American Express' name as of 2022?
2,1,novel-generated,"At the Pepsico AGM held on May 3, 2023, what was the outcome of the shareholder vote on the shareholder proposal for a congruency report by Pepsico on net-zero emissions policies?"
3,1,novel-generated,"As of FY2023Q1, why did Pepsico raise full year guidance for FY2023?"
4,8,novel-generated,What drove the increase in Ulta Beauty's merchandise inventories balance at end of FY2023?


## 5. Chunk size and prefix

The cell below repeats the evaluation for three chunk sizes, each with and without the prefix, to check whether the settings from section 2 are a reasonable choice.

In [7]:
rows = []
for size, overlap in [(150, 30), (250, 50), (400, 80)]:
    for prefix in [False, True]:
        trial = BM25Search(build_chunks(pages, doc_info, size=size, overlap=overlap, prefix=prefix))
        trial_results = {q["financebench_id"]: trial.search(q["question"], k=10) for q in dev}
        rows.append({"chunk size": size, "prefix": prefix, **evaluate_retrieval(trial_results, dev)})
pd.DataFrame(rows)

,chunk size,prefix,recall@1,recall@5,mrr@10,doc_recall@5
0,150,False,0.100,0.100,0.109,0.500
1,150,True,0.067,0.167,0.115,0.567
2,250,False,0.100,0.100,0.109,0.433
3,250,True,0.100,0.133,0.111,0.600
4,400,False,0.067,0.100,0.093,0.467
5,400,True,0.100,0.133,0.114,0.600


It is seen that the chunk size does not make a difference in the results (beyond noise) so our choice as 250 is ok.

The prefix raises document recall at every chunk size, so it will stays on. 

**The settings from section 2 (250 words, 50 overlap, prefix) will be kept.**

## 6. Why BM25 misses

The cell below takes one question that BM25 misses and prints how many chunks contain each of its words. The fewer chunks contain a word, the more weight BM25 gives it.

In [8]:
chunk_words = bm25s.tokenize([c["text"] for c in chunks], stopwords="en", return_ids=False, show_progress=False)
chunk_frequency = Counter(word for words in chunk_words for word in set(words))

missed = next(q for q in dev if q["financebench_id"] == "financebench_id_04672")
print("Q:", missed["question"])
print("evidence page:", sorted(gold_pages(missed)))
print("top 3 pages:  ", search.search(missed["question"], k=3), "\n")

question_words = bm25s.tokenize(missed["question"], stopwords="en", return_ids=False, show_progress=False)[0]
pd.Series({word: chunk_frequency[word] for word in question_words}, name="chunks containing the word").sort_values().to_frame()

Q: Assume that you are a public equities analyst. Answer the following question by primarily using information that is shown in the balance sheet: what is the year end FY2018 net PPNE for 3M? Answer in USD billions.
evidence page: [('3M_2018_10K', 57)]
top 3 pages:   [('MGMRESORTS_2018_10K', 10), ('PEPSICO_2023Q1_EARNINGS', 2), ('MGMRESORTS_2020_10K', 13)] 



,chunks containing the word
ppne,0
analyst,24
answer,32
usd,78
question,81
equities,94
assume,154
billions,156
shown,289
what,295


The table below shows which of the question's words appear on the top 3 pages and on the evidence page, and the rank BM25 gives to each.

In [11]:
words_by_page = {}
for chunk, words in zip(chunks, chunk_words):
    words_by_page.setdefault((chunk["doc_name"], chunk["page_num"]), set()).update(words)

ranking = search.search(missed["question"], k=len(pages), n_chunks=len(chunks))
evidence = sorted(gold_pages(missed))[0]
pd.DataFrame([
    {
        "page": f"{page[0]} p{page[1]}",
        "rank": ranking.index(page) + 1,
        "question words on the page": ", ".join(sorted(w for w in set(question_words) if w in words_by_page[page])),
    }
    for page in ranking[:3] + [evidence]
])

,page,rank,question words on the page
0,MGMRESORTS_2018_10K p10,1,"analyst, assume, fy2018, information, public, you"
1,PEPSICO_2023Q1_EARNINGS p2,2,"answer, net, question, year"
2,MGMRESORTS_2020_10K p13,3,"analyst, assume, information, public, you"
3,3M_2018_10K p57,1304,"3m, balance, fy2018, net"


The cell below prints how the evidence page writes "PPNE", and the share of digits in its text compared with all pages.

In [12]:
page_text = {(p["doc_name"], p["page_num"]): p["text"] for p in pages}
evidence_text = page_text[evidence]
print([line for line in evidence_text.splitlines() if "property" in line.lower()])


def digit_share(text):
    """Share of digits among all letters and digits in a text."""
    digits = sum(ch.isdigit() for ch in text)
    return digits / (digits + sum(ch.isalpha() for ch in text))


print(f"digit share on the evidence page: {digit_share(evidence_text):.2f}")
print(f"digit share across all pages: {digit_share(''.join(p['text'] for p in pages)):.2f}")

['Property, plant and equipment', 'Property, plant and equipment — net']
digit share on the evidence page: 0.23
digit share across all pages: 0.05


The tables above show three problems with this question:

- - Apart from "ppne", the rarest words in the question all come from its instructions, "analyst" (24 chunks), "answer" (32), "usd" (78) and "question" (81). BM25 weights rare words highest, and all three top pages contain some of these words, while the evidence page contains none.
- "ppne" appears in no chunk. The evidence page writes it out as "Property, plant and equipment".
- The evidence page shares only 4 words with the question and is ranked 1304th. Its text is 23% digits against 5% across all pages, so it has fewer words that could match.

These problems might be solved in the next notebooks with different methods of search.

## 7. Saving the results

The top 10 pages per dev question are saved as `results/dev_retrieval_bm25.json`. Each question id maps to a ranked list of `[doc_name, page_num]` pairs, with 0-based page numbers.

In [9]:
out_path = REPO / "results" / "dev_retrieval_bm25.json"
out_path.parent.mkdir(exist_ok=True)
out_path.write_text(json.dumps(results) + "\n")
print("wrote", out_path)

wrote /Users/dogamentese/financebench-rag/results/dev_retrieval_bm25.json


## Summary of results:

- BM25 finds an evidence page in the top 5 for 4 of the 30 dev questions (recall@5 of 0.13), and a page from the right report for 18 of them (doc_recall@5 of 0.60). Finding the report is much easier than finding the page inside it (section 4).
- None of the 10 metrics questions has its evidence page in the top 10. The 5 questions that are found each name a specific filing, event or item, and 4 of them are novel-generated questions (section 4).
- In the 3M example, instruction words such as "analyst" and "answer" are among the rarest words of the question, and the top pages are ones that contain them, while the evidence page shares only 4 words with the question (section 6).
- The prefix with company and year raises doc_recall@5 at every chunk size. Chunk size makes no measurable difference on 30 questions (section 5).
- Kobeissi and Langlais (2026) report the same gap between finding the report and finding the page on this dataset.

The settings carried forward are chunks of 250 words with 50 words overlap and the company, document type and year prefix.